# 🧪 Unit Testing in Python

This notebook covers unit testing best practices using:
- `unittest` — Python's built-in testing framework
- `pytest` — popular third-party testing framework
- Mocking with `unittest.mock`
- Fixtures and parametrization
- Test coverage basics
- Testing async code
- Best practices and anti-patterns


## 1. The Code Under Test
We'll use this simple module throughout the notebook.

In [ ]:
# ---- code_under_test.py (simulated inline) ----

class BankAccount:
    """A simple bank account."""

    def __init__(self, owner: str, balance: float = 0.0):
        if balance < 0:
            raise ValueError('Initial balance cannot be negative')
        self.owner = owner
        self._balance = balance
        self._transactions: list[float] = []

    @property
    def balance(self) -> float:
        return self._balance

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError(f'Deposit amount must be positive, got {amount}')
        self._balance += amount
        self._transactions.append(amount)

    def withdraw(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError(f'Withdrawal amount must be positive, got {amount}')
        if amount > self._balance:
            raise ValueError(f'Insufficient funds: balance={self._balance}, requested={amount}')
        self._balance -= amount
        self._transactions.append(-amount)

    def get_statement(self) -> list[float]:
        return list(self._transactions)


def calculate_interest(principal: float, rate: float, years: int) -> float:
    """Compound interest: A = P(1 + r)^t"""
    if principal < 0 or rate < 0 or years < 0:
        raise ValueError('All arguments must be non-negative')
    return round(principal * (1 + rate) ** years, 2)


def is_valid_email(email: str) -> bool:
    """Basic email validation."""
    import re
    pattern = r'^[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, email))


print('Module loaded successfully')

## 2. unittest — Basic Test Case

In [ ]:
import unittest


class TestBankAccountBasic(unittest.TestCase):
    """Basic tests using unittest.TestCase."""

    # setUp runs BEFORE each test method
    def setUp(self):
        self.account = BankAccount('Alice', 1000.0)

    # tearDown runs AFTER each test method (even if it fails)
    def tearDown(self):
        pass  # Cleanup if needed (close files, DB connections, etc.)

    # ---- Test methods MUST start with test_ ----

    def test_initial_balance(self):
        self.assertEqual(self.account.balance, 1000.0)

    def test_initial_owner(self):
        self.assertEqual(self.account.owner, 'Alice')

    def test_deposit_increases_balance(self):
        self.account.deposit(500.0)
        self.assertEqual(self.account.balance, 1500.0)

    def test_withdraw_decreases_balance(self):
        self.account.withdraw(200.0)
        self.assertEqual(self.account.balance, 800.0)

    def test_deposit_zero_raises_error(self):
        with self.assertRaises(ValueError):
            self.account.deposit(0)

    def test_deposit_negative_raises_error(self):
        with self.assertRaises(ValueError) as ctx:
            self.account.deposit(-100)
        self.assertIn('positive', str(ctx.exception))

    def test_overdraft_raises_error(self):
        with self.assertRaises(ValueError) as ctx:
            self.account.withdraw(9999)
        self.assertIn('Insufficient funds', str(ctx.exception))

    def test_negative_initial_balance_raises(self):
        with self.assertRaises(ValueError):
            BankAccount('Bob', -50)

    def test_statement_records_transactions(self):
        self.account.deposit(500)
        self.account.withdraw(200)
        statement = self.account.get_statement()
        self.assertEqual(statement, [500, -200])


# Run inside notebook
loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestBankAccountBasic)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

## 3. All unittest Assertion Methods

In [ ]:
import unittest


class TestAssertions(unittest.TestCase):
    """Demonstrates the full range of assertion methods."""

    def test_equality(self):
        self.assertEqual(2 + 2, 4)        # a == b
        self.assertNotEqual(2 + 2, 5)     # a != b

    def test_boolean(self):
        self.assertTrue(3 > 2)            # bool(x) is True
        self.assertFalse(2 > 3)           # bool(x) is False

    def test_none(self):
        self.assertIsNone(None)           # x is None
        self.assertIsNotNone(42)          # x is not None

    def test_identity(self):
        x = [1, 2, 3]
        y = x
        self.assertIs(x, y)              # x is y (same object)
        self.assertIsNot(x, [1, 2, 3])  # x is not [1,2,3] (different objects)

    def test_membership(self):
        self.assertIn('a', ['a', 'b', 'c'])      # x in collection
        self.assertNotIn('z', ['a', 'b', 'c'])   # x not in collection

    def test_types(self):
        self.assertIsInstance(42, int)
        self.assertNotIsInstance(42, str)

    def test_comparisons(self):
        self.assertGreater(5, 3)          # a > b
        self.assertGreaterEqual(5, 5)     # a >= b
        self.assertLess(3, 5)             # a < b
        self.assertLessEqual(3, 3)        # a <= b

    def test_floats(self):
        # NEVER use assertEqual for floats — use assertAlmostEqual
        self.assertAlmostEqual(0.1 + 0.2, 0.3, places=10)

    def test_sequences(self):
        self.assertListEqual([1, 2, 3], [1, 2, 3])
        self.assertTupleEqual((1, 2), (1, 2))
        self.assertSetEqual({1, 2, 3}, {3, 1, 2})
        self.assertDictEqual({'a': 1}, {'a': 1})

    def test_strings(self):
        self.assertIn('hello', 'say hello world')   # substring check
        self.assertRegex('hello123', r'\w+\d+')
        self.assertNotRegex('hello', r'^\d+')


loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestAssertions)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## 4. Mocking with `unittest.mock`

In [ ]:
import unittest
from unittest.mock import MagicMock, patch, call


# ---- Code that uses external dependencies ----
class EmailService:
    def send(self, to: str, subject: str, body: str) -> bool:
        # In reality, this would call an SMTP server
        raise NotImplementedError('Real implementation contacts SMTP')

class UserRegistration:
    def __init__(self, email_service: EmailService):
        self.email_service = email_service
        self.users = []

    def register(self, email: str, name: str) -> dict:
        if not is_valid_email(email):
            raise ValueError(f'Invalid email: {email}')
        user = {'email': email, 'name': name}
        self.users.append(user)
        self.email_service.send(
            to=email,
            subject='Welcome!',
            body=f'Hi {name}, welcome aboard!'
        )
        return user


class TestUserRegistration(unittest.TestCase):

    def setUp(self):
        self.mock_email = MagicMock(spec=EmailService)
        self.mock_email.send.return_value = True
        self.registration = UserRegistration(self.mock_email)

    def test_register_adds_user(self):
        self.registration.register('alice@example.com', 'Alice')
        self.assertEqual(len(self.registration.users), 1)
        self.assertEqual(self.registration.users[0]['name'], 'Alice')

    def test_register_sends_welcome_email(self):
        self.registration.register('alice@example.com', 'Alice')
        # Verify the mock was called with correct arguments
        self.mock_email.send.assert_called_once_with(
            to='alice@example.com',
            subject='Welcome!',
            body='Hi Alice, welcome aboard!'
        )

    def test_invalid_email_raises(self):
        with self.assertRaises(ValueError):
            self.registration.register('not-an-email', 'Bob')
        # Confirm email was NOT sent
        self.mock_email.send.assert_not_called()

    def test_register_multiple_users(self):
        self.registration.register('alice@example.com', 'Alice')
        self.registration.register('bob@example.com', 'Bob')
        self.assertEqual(self.mock_email.send.call_count, 2)

    def test_mock_side_effect(self):
        # Make the mock raise an exception on second call
        self.mock_email.send.side_effect = [True, ConnectionError('SMTP down')]
        self.registration.register('alice@example.com', 'Alice')  # OK
        with self.assertRaises(ConnectionError):
            self.registration.register('bob@example.com', 'Bob')  # Raises


loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestUserRegistration)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## 5. `@patch` Decorator — Patching at the Module Level

In [ ]:
import unittest
from unittest.mock import patch, MagicMock
import datetime


def get_greeting():
    hour = datetime.datetime.now().hour
    if hour < 12:
        return 'Good morning!'
    elif hour < 18:
        return 'Good afternoon!'
    else:
        return 'Good evening!'


class TestGreeting(unittest.TestCase):

    @patch('datetime.datetime')
    def test_morning_greeting(self, mock_dt):
        mock_dt.now.return_value.hour = 9  # 9 AM
        self.assertEqual(get_greeting(), 'Good morning!')

    @patch('datetime.datetime')
    def test_afternoon_greeting(self, mock_dt):
        mock_dt.now.return_value.hour = 14  # 2 PM
        self.assertEqual(get_greeting(), 'Good afternoon!')

    @patch('datetime.datetime')
    def test_evening_greeting(self, mock_dt):
        mock_dt.now.return_value.hour = 20  # 8 PM
        self.assertEqual(get_greeting(), 'Good evening!')


# Using patch as context manager
with patch('datetime.datetime') as mock_dt:
    mock_dt.now.return_value.hour = 6
    print(f'Mock result: {get_greeting()}')  # Good morning!


loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestGreeting)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## 6. Parametrized Testing (pytest style)

In [ ]:
import unittest


class TestCalculateInterest(unittest.TestCase):
    """Test multiple inputs using a loop (poor man's parametrize)."""

    test_cases = [
        # (principal, rate, years, expected)
        (1000, 0.05, 1, 1050.0),
        (1000, 0.05, 2, 1102.5),
        (1000, 0.10, 3, 1331.0),
        (500,  0.0,  5, 500.0),   # 0% interest
        (0,    0.10, 10, 0.0),    # 0 principal
    ]

    def test_interest_calculations(self):
        for principal, rate, years, expected in self.test_cases:
            with self.subTest(principal=principal, rate=rate, years=years):
                result = calculate_interest(principal, rate, years)
                self.assertAlmostEqual(result, expected, places=2,
                    msg=f'Failed for P={principal}, r={rate}, t={years}')

    def test_invalid_inputs(self):
        invalid_cases = [
            (-100, 0.05, 1),   # negative principal
            (100, -0.05, 1),   # negative rate
            (100, 0.05, -1),   # negative years
        ]
        for principal, rate, years in invalid_cases:
            with self.subTest(principal=principal, rate=rate, years=years):
                with self.assertRaises(ValueError):
                    calculate_interest(principal, rate, years)


loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestCalculateInterest)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## 7. Testing with pytest (run as script)

In [ ]:
# Write pytest-style tests to a file and run them
import subprocess, sys, os

pytest_code = '''
import pytest
import re

# ---- Code under test (inline) ----
class BankAccount:
    def __init__(self, owner, balance=0.0):
        if balance < 0:
            raise ValueError("Initial balance cannot be negative")
        self.owner = owner
        self._balance = balance

    @property
    def balance(self): return self._balance

    def deposit(self, amount):
        if amount <= 0: raise ValueError("must be positive")
        self._balance += amount

    def withdraw(self, amount):
        if amount <= 0: raise ValueError("must be positive")
        if amount > self._balance: raise ValueError("Insufficient funds")
        self._balance -= amount

def is_valid_email(email):
    return bool(re.match(r"^[\\w.+-]+@[\\w-]+\\.[a-zA-Z]{2,}$", email))

# ---- Fixtures ----
@pytest.fixture
def account():
    """Fresh BankAccount for each test."""
    return BankAccount("Alice", 1000.0)

@pytest.fixture
def empty_account():
    return BankAccount("Bob", 0.0)

# ---- Tests using fixtures ----
def test_initial_balance(account):
    assert account.balance == 1000.0

def test_deposit(account):
    account.deposit(500)
    assert account.balance == 1500.0

def test_withdraw(account):
    account.withdraw(300)
    assert account.balance == 700.0

def test_overdraft_raises(account):
    with pytest.raises(ValueError, match="Insufficient funds"):
        account.withdraw(9999)

def test_empty_account_cannot_withdraw(empty_account):
    with pytest.raises(ValueError):
        empty_account.withdraw(1)

# ---- Parametrized tests ----
@pytest.mark.parametrize("email,expected", [
    ("user@example.com", True),
    ("user.name+tag@sub.domain.co", True),
    ("not-an-email", False),
    ("missing@domain", False),
    ("@nodomain.com", False),
    ("", False),
])
def test_email_validation(email, expected):
    assert is_valid_email(email) == expected

# ---- Exception messages ----
def test_negative_deposit_message(account):
    with pytest.raises(ValueError, match="positive"):
        account.deposit(-50)
'''

with open('/tmp/test_bank.py', 'w') as f:
    f.write(pytest_code)

# Install pytest if needed
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pytest', '-q'], capture_output=True)

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '/tmp/test_bank.py', '-v', '--tb=short'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:500])

## 8. Testing Async Code

In [ ]:
import asyncio
import unittest
from unittest.mock import AsyncMock, patch


# ---- Async code under test ----
async def fetch_price(symbol: str) -> float:
    """Simulates calling an async market data API."""
    await asyncio.sleep(0.01)  # Simulated network delay
    prices = {'AAPL': 189.50, 'GOOG': 175.30, 'MSFT': 420.00}
    if symbol not in prices:
        raise ValueError(f'Unknown symbol: {symbol}')
    return prices[symbol]

async def get_portfolio_value(symbols: list[str]) -> float:
    """Fetch all prices concurrently and sum them."""
    prices = await asyncio.gather(*[fetch_price(s) for s in symbols])
    return sum(prices)


# ---- Async tests ----
class TestAsyncFunctions(unittest.IsolatedAsyncioTestCase):
    """Use IsolatedAsyncioTestCase for async tests (Python 3.8+)."""

    async def test_fetch_known_symbol(self):
        price = await fetch_price('AAPL')
        self.assertEqual(price, 189.50)

    async def test_fetch_unknown_symbol_raises(self):
        with self.assertRaises(ValueError) as ctx:
            await fetch_price('FAKE')
        self.assertIn('FAKE', str(ctx.exception))

    async def test_portfolio_value(self):
        total = await get_portfolio_value(['AAPL', 'GOOG'])
        self.assertAlmostEqual(total, 189.50 + 175.30, places=2)

    async def test_mocked_async_call(self):
        with patch('__main__.fetch_price', new_callable=AsyncMock) as mock_fetch:
            mock_fetch.return_value = 999.99
            total = await get_portfolio_value(['X', 'Y'])
            self.assertEqual(total, 1999.98)
            self.assertEqual(mock_fetch.call_count, 2)


loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestAsyncFunctions)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## 9. Test Isolation & Anti-Patterns

In [ ]:
import unittest

# ========== ❌ BAD: Tests depend on each other ==========
class BadTestSuite(unittest.TestCase):
    shared_account = None  # Shared mutable state — DANGER!

    def test_01_create(self):
        BadTestSuite.shared_account = BankAccount('Eve', 500)
        self.assertIsNotNone(self.shared_account)

    def test_02_deposit(self):
        # This RELIES on test_01 having run first — fragile!
        self.shared_account.deposit(100)
        self.assertEqual(self.shared_account.balance, 600)

    def test_03_withdraw(self):
        # Relies on both test_01 and test_02 — even more fragile!
        self.shared_account.withdraw(200)
        self.assertEqual(self.shared_account.balance, 400)


# ========== ✅ GOOD: Each test is fully independent ==========
class GoodTestSuite(unittest.TestCase):

    def setUp(self):
        # Fresh state for EVERY test — isolation guaranteed
        self.account = BankAccount('Eve', 500)

    def test_deposit(self):
        self.account.deposit(100)
        self.assertEqual(self.account.balance, 600)

    def test_withdraw(self):
        # Starts fresh at 500 — not affected by test_deposit
        self.account.withdraw(200)
        self.assertEqual(self.account.balance, 300)

    def test_deposit_then_withdraw(self):
        # Tests the combination in isolation
        self.account.deposit(100)
        self.account.withdraw(200)
        self.assertEqual(self.account.balance, 400)


print('=== Good Test Suite ===')
loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(GoodTestSuite)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## 10. Test Coverage

Coverage measures what percentage of your code is exercised by tests.

In [ ]:
import subprocess, sys

# Install coverage
subprocess.run([sys.executable, '-m', 'pip', 'install', 'coverage', '-q'], capture_output=True)

coverage_test = '''
# module_to_test.py
def grade(score):
    if score >= 90: return 'A'
    elif score >= 80: return 'B'
    elif score >= 70: return 'C'
    elif score >= 60: return 'D'
    else: return 'F'

import unittest

class TestGrade(unittest.TestCase):
    # Only testing A and F — low coverage!
    def test_a(self): self.assertEqual(grade(95), 'A')
    def test_f(self): self.assertEqual(grade(50), 'F')
    # Uncomment to increase coverage:
    # def test_b(self): self.assertEqual(grade(85), 'B')
    # def test_c(self): self.assertEqual(grade(75), 'C')
    # def test_d(self): self.assertEqual(grade(65), 'D')

if __name__ == '__main__':
    unittest.main()
'''

with open('/tmp/coverage_demo.py', 'w') as f:
    f.write(coverage_test)

result = subprocess.run(
    [sys.executable, '-m', 'coverage', 'run', '/tmp/coverage_demo.py'],
    capture_output=True, text=True, cwd='/tmp'
)

report = subprocess.run(
    [sys.executable, '-m', 'coverage', 'report', '--include=/tmp/coverage_demo.py'],
    capture_output=True, text=True, cwd='/tmp'
)

print(report.stdout)
print()
print('Tips for good coverage:')
print('  • Aim for 80-90%+ line coverage as a baseline')
print('  • 100% coverage does not mean 0 bugs — test edge cases!')
print('  • Use: coverage html  to generate a visual HTML report')

## Summary

| Concept | Tool | When to use |
|---|---|---|
| Basic unit tests | `unittest.TestCase` | Always |
| Modern tests | `pytest` | Preferred for new projects |
| Replace external deps | `MagicMock`, `patch` | DB, HTTP, SMTP, time |
| Multiple inputs | `subTest`, `@parametrize` | Validating ranges of values |
| Async functions | `IsolatedAsyncioTestCase` | async/await code |
| Coverage | `coverage.py` | Measure test completeness |
| Isolation | `setUp`/`tearDown` | Always — never share state |